20 epochs

In [1]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-HPO-8B-Term-ID-20-49769eb1",  # Your fine-tuned HPO model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Human Phenotype Ontology (HPO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since HPO IDs are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"HPO_ID": "Error"})

# Improved function to extract JSON from the LLM response for HPO data
def extract_hpo_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'HPO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"HPO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract HPO_ID value directly
        hpo_id_match = re.search(r'"HPO_ID"[\s:]*"([^"]+)"', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        hpo_id_match = re.search(r'HPO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"HPO_ID": "None"}

        # If all parsing attempts fail
        return {"HPO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"HPO_ID": "Parse_Error"}

# Enhanced prompt to get HPO ID for a given HPO term
def get_hpo_id_from_term(hpo_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Human Phenotype Ontology term "{hpo_term}", provide the corresponding HPO ID.

Instructions:
- Provide the EXACT official HPO ID as it appears in the Human Phenotype Ontology database
- Choose the most direct, standard HPO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary phenotype IDs over highly specific subphenotypes)
- Return the primary, commonly used HPO ID
- HPO IDs follow the format: HP:XXXXXXX (where X are 7 digits)
- Examples: 
  - "All" → HP:0000001
  - "Phenotypic abnormality" → HP:0000118
  - "Growth abnormality" → HP:0001507
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "HPO_ID": "<hpo_id_here>"
}}

HPO Term: {hpo_term}"""

    response_text = query_together(prompt)
    result = extract_hpo_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "hpo_term": hpo_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("hpo_term_to_id_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("HPO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_hpo_id, new_hpo_id):
    """
    Calculate match result between original and new HPO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_hpo_id) or pd.isna(new_hpo_id):
        return 0
    return int(str(original_hpo_id).strip() == str(new_hpo_id).strip())

# Main processing function
def main():
    print("Starting HPO term to ID mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("hpo_term_to_id_api_responses_finetuned.jsonl"):
        with open("hpo_term_to_id_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample HPO term
    print("Testing API connection...")
    test_result = get_hpo_id_from_term("Seizure")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the HPO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_hpo_id"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            hpo_term = row["hpo_term"]  # Using the hpo_term column from the CSV
            original_hpo_id = row["hpo_id"]  # Original HPO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {hpo_term}")

            start_time = time.time()
            new_hpo_id = get_hpo_id_from_term(hpo_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new HPO ID
            df.loc[idx, "new_hpo_id"] = new_hpo_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_hpo_id, new_hpo_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_hpo_id}, New: {new_hpo_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_hpo_term_to_id_results_progress20.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_hpo_id"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_hpo_term_to_id_results_final20.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Term to ID) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_hpo_id"] == "None").sum()
        error_results = (df["new_hpo_id"] == "Error").sum()
        parse_error_results = (df["new_hpo_id"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid HPO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_hpo_id"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about HPO annotations if available
        if 'hpo_annotations' in df.columns:
            print(f"\nBreakdown by HPO Annotations:")
            # Create bins for HPO annotations
            df['annotation_bins'] = pd.cut(df['hpo_annotations'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
            annotation_stats = df.groupby('annotation_bins').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(annotation_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_hpo_term_to_id_results_final5.csv")
    print(f"Progress file: finetuned_hpo_term_to_id_results_progress5.csv")
    print(f"Log file: hpo_term_to_id_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, hpo_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting HPO term to ID mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: HP:0001250
API test successful, proceeding with batch processing...
Loaded 18800 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv
Columns in the dataset: ['hpo_id', 'hpo_term', 'hpo_parent', 'hpo_term_pmc', 'hpo_annotations', 'hpo_id_pmc', 'normalized_term', 'normalized_id', 'hpo_match']
Processing 1/18800: Seizure
Sending request (attempt 1/5)...
  Original: HP:0001250, New: HP:0001250, Match: 1
Progress saved. Processed 1/18800 terms.
Waiting 2.76 seconds before next request...
Processing 2/18800: Global developmental delay
Sending request (attempt 1/5)...
  Original: HP:0001263, New: HP:0001263, Match: 1
Waiting 3.63 seconds before next request...
Processing 3/18800: Anti-AK5 antibody positivity
Sending request (attempt 1/5)...
  Original: HP:5000000, New: HP:0004444, Match:

C:\Users\sp5526s\AppData\Local\Temp\ipykernel_29816\809807288.py:280: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  annotation_stats = df.groupby('annotation_bins').agg({


15 epochs

In [2]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-HPO-8B-Term-ID-15-bdcabb4a",  # Your fine-tuned HPO model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Human Phenotype Ontology (HPO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since HPO IDs are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"HPO_ID": "Error"})

# Improved function to extract JSON from the LLM response for HPO data
def extract_hpo_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'HPO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"HPO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract HPO_ID value directly
        hpo_id_match = re.search(r'"HPO_ID"[\s:]*"([^"]+)"', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        hpo_id_match = re.search(r'HPO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"HPO_ID": "None"}

        # If all parsing attempts fail
        return {"HPO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"HPO_ID": "Parse_Error"}

# Enhanced prompt to get HPO ID for a given HPO term
def get_hpo_id_from_term(hpo_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Human Phenotype Ontology term "{hpo_term}", provide the corresponding HPO ID.

Instructions:
- Provide the EXACT official HPO ID as it appears in the Human Phenotype Ontology database
- Choose the most direct, standard HPO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary phenotype IDs over highly specific subphenotypes)
- Return the primary, commonly used HPO ID
- HPO IDs follow the format: HP:XXXXXXX (where X are 7 digits)
- Examples: 
  - "All" → HP:0000001
  - "Phenotypic abnormality" → HP:0000118
  - "Growth abnormality" → HP:0001507
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "HPO_ID": "<hpo_id_here>"
}}

HPO Term: {hpo_term}"""

    response_text = query_together(prompt)
    result = extract_hpo_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "hpo_term": hpo_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("hpo_term_to_id_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("HPO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_hpo_id, new_hpo_id):
    """
    Calculate match result between original and new HPO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_hpo_id) or pd.isna(new_hpo_id):
        return 0
    return int(str(original_hpo_id).strip() == str(new_hpo_id).strip())

# Main processing function
def main():
    print("Starting HPO term to ID mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("hpo_term_to_id_api_responses_finetuned.jsonl"):
        with open("hpo_term_to_id_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample HPO term
    print("Testing API connection...")
    test_result = get_hpo_id_from_term("Seizure")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the HPO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_hpo_id"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            hpo_term = row["hpo_term"]  # Using the hpo_term column from the CSV
            original_hpo_id = row["hpo_id"]  # Original HPO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {hpo_term}")

            start_time = time.time()
            new_hpo_id = get_hpo_id_from_term(hpo_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new HPO ID
            df.loc[idx, "new_hpo_id"] = new_hpo_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_hpo_id, new_hpo_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_hpo_id}, New: {new_hpo_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_hpo_term_to_id_results_progress15.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_hpo_id"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_hpo_term_to_id_results_final15.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Term to ID) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_hpo_id"] == "None").sum()
        error_results = (df["new_hpo_id"] == "Error").sum()
        parse_error_results = (df["new_hpo_id"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid HPO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_hpo_id"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about HPO annotations if available
        if 'hpo_annotations' in df.columns:
            print(f"\nBreakdown by HPO Annotations:")
            # Create bins for HPO annotations
            df['annotation_bins'] = pd.cut(df['hpo_annotations'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
            annotation_stats = df.groupby('annotation_bins').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(annotation_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_hpo_term_to_id_results_final5.csv")
    print(f"Progress file: finetuned_hpo_term_to_id_results_progress5.csv")
    print(f"Log file: hpo_term_to_id_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, hpo_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting HPO term to ID mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: HP:0001250
API test successful, proceeding with batch processing...
Loaded 18800 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv
Columns in the dataset: ['hpo_id', 'hpo_term', 'hpo_parent', 'hpo_term_pmc', 'hpo_annotations', 'hpo_id_pmc', 'normalized_term', 'normalized_id', 'hpo_match']
Processing 1/18800: Seizure
Sending request (attempt 1/5)...
  Original: HP:0001250, New: HP:0001250, Match: 1
Progress saved. Processed 1/18800 terms.
Waiting 3.29 seconds before next request...
Processing 2/18800: Global developmental delay
Sending request (attempt 1/5)...
  Original: HP:0001263, New: HP:0001263, Match: 1
Waiting 3.96 seconds before next request...


KeyboardInterrupt: 

10 epoch

In [ ]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = ""
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-HPO-8B-Term-ID-10-eb98b39d",  # Your fine-tuned HPO model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Human Phenotype Ontology (HPO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since HPO IDs are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"HPO_ID": "Error"})

# Improved function to extract JSON from the LLM response for HPO data
def extract_hpo_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'HPO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"HPO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract HPO_ID value directly
        hpo_id_match = re.search(r'"HPO_ID"[\s:]*"([^"]+)"', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        hpo_id_match = re.search(r'HPO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"HPO_ID": "None"}

        # If all parsing attempts fail
        return {"HPO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"HPO_ID": "Parse_Error"}

# Enhanced prompt to get HPO ID for a given HPO term
def get_hpo_id_from_term(hpo_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Human Phenotype Ontology term "{hpo_term}", provide the corresponding HPO ID.

Instructions:
- Provide the EXACT official HPO ID as it appears in the Human Phenotype Ontology database
- Choose the most direct, standard HPO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary phenotype IDs over highly specific subphenotypes)
- Return the primary, commonly used HPO ID
- HPO IDs follow the format: HP:XXXXXXX (where X are 7 digits)
- Examples: 
  - "All" → HP:0000001
  - "Phenotypic abnormality" → HP:0000118
  - "Growth abnormality" → HP:0001507
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "HPO_ID": "<hpo_id_here>"
}}

HPO Term: {hpo_term}"""

    response_text = query_together(prompt)
    result = extract_hpo_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "hpo_term": hpo_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("hpo_term_to_id_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("HPO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_hpo_id, new_hpo_id):
    """
    Calculate match result between original and new HPO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_hpo_id) or pd.isna(new_hpo_id):
        return 0
    return int(str(original_hpo_id).strip() == str(new_hpo_id).strip())

# Main processing function
def main():
    print("Starting HPO term to ID mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("hpo_term_to_id_api_responses_finetuned.jsonl"):
        with open("hpo_term_to_id_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample HPO term
    print("Testing API connection...")
    test_result = get_hpo_id_from_term("Seizure")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the HPO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_hpo_id"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            hpo_term = row["hpo_term"]  # Using the hpo_term column from the CSV
            original_hpo_id = row["hpo_id"]  # Original HPO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {hpo_term}")

            start_time = time.time()
            new_hpo_id = get_hpo_id_from_term(hpo_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new HPO ID
            df.loc[idx, "new_hpo_id"] = new_hpo_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_hpo_id, new_hpo_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_hpo_id}, New: {new_hpo_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_hpo_term_to_id_results_progress10.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_hpo_id"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_hpo_term_to_id_results_final10.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Term to ID) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_hpo_id"] == "None").sum()
        error_results = (df["new_hpo_id"] == "Error").sum()
        parse_error_results = (df["new_hpo_id"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid HPO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_hpo_id"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about HPO annotations if available
        if 'hpo_annotations' in df.columns:
            print(f"\nBreakdown by HPO Annotations:")
            # Create bins for HPO annotations
            df['annotation_bins'] = pd.cut(df['hpo_annotations'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
            annotation_stats = df.groupby('annotation_bins').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(annotation_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_hpo_term_to_id_results_final5.csv")
    print(f"Progress file: finetuned_hpo_term_to_id_results_progress5.csv")
    print(f"Log file: hpo_term_to_id_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, hpo_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting HPO term to ID mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Request error: Error code: 503 - The server is overloaded or not ready yet.
Attempt 2/5 - Waiting 2.68 seconds before retry...
Sending request (attempt 2/5)...
Request error: Error code: 503 - The server is overloaded or not ready yet.
Attempt 3/5 - Waiting 4.75 seconds before retry...
Sending request (attempt 3/5)...
Request error: Error code: 503 - The server is overloaded or not ready yet.
Attempt 4/5 - Waiting 8.45 seconds before retry...
Sending request (attempt 4/5)...
Request error: Error code: 503 - The server is overloaded or not ready yet.
Attempt 5/5 - Waiting 19.14 seconds before retry...
Sending request (attempt 5/5)...
Request error: Error code: 503 - The server is overloaded or not ready yet.
Test result: Error
API test failed. Please check your API key and connection.


In [5]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_MpajHfgg8br5MUQBjyoc2xmUrzc8mKwHlmo0TXvXH7s"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-HPO-8B-Term-ID-10-8685e921",  # Your fine-tuned HPO model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Human Phenotype Ontology (HPO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since HPO IDs are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"HPO_ID": "Error"})

# Improved function to extract JSON from the LLM response for HPO data
def extract_hpo_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'HPO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"HPO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'HPO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract HPO_ID value directly
        hpo_id_match = re.search(r'"HPO_ID"[\s:]*"([^"]+)"', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        hpo_id_match = re.search(r'HPO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if hpo_id_match:
            return {"HPO_ID": hpo_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"HPO_ID": "None"}

        # If all parsing attempts fail
        return {"HPO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"HPO_ID": "Parse_Error"}

# Enhanced prompt to get HPO ID for a given HPO term
def get_hpo_id_from_term(hpo_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Human Phenotype Ontology term "{hpo_term}", provide the corresponding HPO ID.

Instructions:
- Provide the EXACT official HPO ID as it appears in the Human Phenotype Ontology database
- Choose the most direct, standard HPO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary phenotype IDs over highly specific subphenotypes)
- Return the primary, commonly used HPO ID
- HPO IDs follow the format: HP:XXXXXXX (where X are 7 digits)
- Examples: 
  - "All" → HP:0000001
  - "Phenotypic abnormality" → HP:0000118
  - "Growth abnormality" → HP:0001507
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "HPO_ID": "<hpo_id_here>"
}}

HPO Term: {hpo_term}"""

    response_text = query_together(prompt)
    result = extract_hpo_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "hpo_term": hpo_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("hpo_term_to_id_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("HPO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_hpo_id, new_hpo_id):
    """
    Calculate match result between original and new HPO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_hpo_id) or pd.isna(new_hpo_id):
        return 0
    return int(str(original_hpo_id).strip() == str(new_hpo_id).strip())

# Main processing function
def main():
    print("Starting HPO term to ID mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("hpo_term_to_id_api_responses_finetuned.jsonl"):
        with open("hpo_term_to_id_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample HPO term
    print("Testing API connection...")
    test_result = get_hpo_id_from_term("Seizure")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the HPO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_hpo_id"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            hpo_term = row["hpo_term"]  # Using the hpo_term column from the CSV
            original_hpo_id = row["hpo_id"]  # Original HPO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {hpo_term}")

            start_time = time.time()
            new_hpo_id = get_hpo_id_from_term(hpo_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new HPO ID
            df.loc[idx, "new_hpo_id"] = new_hpo_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_hpo_id, new_hpo_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_hpo_id}, New: {new_hpo_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_hpo_term_to_id_results_progress10.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_hpo_id"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'hpo_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_hpo_term_to_id_results_final10.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Term to ID) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_hpo_id"] == "None").sum()
        error_results = (df["new_hpo_id"] == "Error").sum()
        parse_error_results = (df["new_hpo_id"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid HPO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_hpo_id"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about HPO annotations if available
        if 'hpo_annotations' in df.columns:
            print(f"\nBreakdown by HPO Annotations:")
            # Create bins for HPO annotations
            df['annotation_bins'] = pd.cut(df['hpo_annotations'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
            annotation_stats = df.groupby('annotation_bins').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(annotation_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_hpo_term_to_id_results_final5.csv")
    print(f"Progress file: finetuned_hpo_term_to_id_results_progress5.csv")
    print(f"Log file: hpo_term_to_id_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, hpo_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting HPO term to ID mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: HP:0001250
API test successful, proceeding with batch processing...
Loaded 18800 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\hpo_terms.csv
Columns in the dataset: ['hpo_id', 'hpo_term', 'hpo_parent', 'hpo_term_pmc', 'hpo_annotations', 'hpo_id_pmc', 'normalized_term', 'normalized_id', 'hpo_match']
Processing 1/18800: Seizure
Sending request (attempt 1/5)...
  Original: HP:0001250, New: HP:0001250, Match: 1
Progress saved. Processed 1/18800 terms.
Waiting 3.47 seconds before next request...
Processing 2/18800: Global developmental delay
Sending request (attempt 1/5)...
  Original: HP:0001263, New: HP:0001263, Match: 1
Waiting 2.33 seconds before next request...
Processing 3/18800: Anti-AK5 antibody positivity
Sending request (attempt 1/5)...
  Original: HP:5000000, New: HP:0004443, Match:

C:\Users\sp5526s\AppData\Local\Temp\ipykernel_29816\3387101567.py:280: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  annotation_stats = df.groupby('annotation_bins').agg({
